# Fase 1: Fine-tuning de BERT en GoEmotions

Entrenamiento de BERT base para clasificación multi-label de 28 emociones.
Este modelo sirve como baseline para las fases posteriores de compresión.

In [1]:
# Instalación de dependencias (para Google Colab)
# !pip install transformers datasets torch scikit-learn accelerate seaborn

In [ ]:
import sys
import os

# En Colab, montar Drive y ajustar path al proyecto
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'

# En local
#PROJECT_ROOT = "/Users/guido.biosca/Personal/transformer-structural-compression-v2/"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

: 

In [11]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import TrainingArguments, Trainer
from src.data import load_goemotions
from src.models import load_bert_classifier
from src.utils import compute_metrics

## 1. Cargar y explorar el dataset

In [12]:
train_ds, val_ds, test_ds, emotion_names, data_collator = load_goemotions()

print(f"Train: {len(train_ds)} ejemplos")
print(f"Validation: {len(val_ds)} ejemplos")
print(f"Test: {len(test_ds)} ejemplos")
print(f"\nEmociones ({len(emotion_names)}): {emotion_names}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

...

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Train: 43410 ejemplos
Validation: 5426 ejemplos
Test: 5427 ejemplos

Emociones (28): ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [ ]:
# Distribución de emociones en el conjunto de entrenamiento
train_labels = np.array(train_ds["labels"])
label_counts = train_labels.sum(axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.barh(emotion_names, label_counts)
ax.set_xlabel("Número de ejemplos")
ax.set_title("Distribución de emociones en el conjunto de entrenamiento")
plt.tight_layout()
plt.show()

## 2. Cargar modelo

In [ ]:
model = load_bert_classifier()
print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Configurar entrenamiento

In [ ]:
output_dir = os.path.join(PROJECT_ROOT, "results", "bert-goemotions")

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    report_to="none",
    seed=42,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

## 4. Entrenar

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 5. Evaluar en test set

In [ ]:
test_metrics = trainer.evaluate(test_ds)

print(f"\nResultados en test:")
print(f"  F1 macro: {test_metrics['eval_f1_macro']:.4f}")
print(f"  F1 micro: {test_metrics['eval_f1_micro']:.4f}")

In [ ]:
# F1 por emoción
f1_per_emotion = {}
for i, name in enumerate(emotion_names):
    key = f"eval_f1_label_{i}"
    f1_per_emotion[name] = test_metrics[key]

f1_df = pd.DataFrame.from_dict(f1_per_emotion, orient="index", columns=["F1"])
f1_df = f1_df.sort_values("F1", ascending=True)
print("\nF1 por emoción:")
print(f1_df.to_string())

## 6. Guardar modelo y métricas

In [ ]:
# Guardar modelo final
model_path = os.path.join(PROJECT_ROOT, "results", "bert-goemotions-final")
trainer.save_model(model_path)
print(f"Modelo guardado en: {model_path}")

# Guardar métricas
metrics_path = os.path.join(PROJECT_ROOT, "results", "baseline_metrics.json")
metrics_to_save = {
    "f1_macro": test_metrics["eval_f1_macro"],
    "f1_micro": test_metrics["eval_f1_micro"],
    "f1_per_emotion": f1_per_emotion,
}
with open(metrics_path, "w") as f:
    json.dump(metrics_to_save, f, indent=2)
print(f"Métricas guardadas en: {metrics_path}")

## 7. Visualización de resultados

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette("viridis", len(f1_df))
ax.barh(f1_df.index, f1_df["F1"], color=colors)
ax.set_xlabel("F1 Score")
ax.set_title("F1 Score por emoción (test set) — BERT base baseline")
ax.axvline(x=test_metrics["eval_f1_macro"], color="red", linestyle="--",
           label=f"F1 macro = {test_metrics['eval_f1_macro']:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "f1_per_emotion.png"), dpi=150)
plt.show()
print("Gráfica guardada en results/f1_per_emotion.png")